# 00 — Prepare shared SAM masks once

Run this notebook before H1, H2 and H3. It keeps the same frozen **54 landmarks**, using
**all eligible distinct photographs up to 50 per landmark** and its assigned architectural anchor.
Landmarks with fewer eligible views keep their actual count; none are replaced or padded.
It imports compatible saved masks and generates only missing ones. It runs no reconstruction
experiments and selects no intervention strengths. The 79 final holdout landmarks stay untouched.

1. Keep this notebook and its matching source ZIP in `MyDrive/Point3_Hypothesis54`.
2. Select a GPU runtime. G4 uses a separate CUDA 12.8 environment; A100/L4/T4 use the existing profile.
3. Add `HF_TOKEN` in Colab Secrets, enable access for this notebook, and use approved SAM 3 access.
4. Run cells in order. Wait for **READY: 54 landmarks / N masks**, and inspect the overlays.
5. Start H1, H2 and H3 on separate GPU sessions, or sequentially on one GPU. All three only read masks.

Run only one copy of mask preparation against this cache at a time. Progress persists in Drive;
rerun with the same settings after interruption. The eight-hour session limit is not a runtime estimate.
Masks are checkpointed after each ten newly processed images. Existing files are searched read-only.
Earlier masks require matching RGB content, anchor and SAM settings; filenames alone are insufficient.
Unknown-provenance files appear in the discovery report, without silently becoming experimental inputs.

Fixed parameters: seed **1729**, SAM confidence **0.5**, pinned SAM revision and deterministic
selection capped at 50. The up-to-50 rule was explicitly authorized on 2026-11-15 after encountering
smaller scenes. Actual counts are saved in `view_counts.csv` and all hypothesis results.
Changing settings can require new masks. Forth Bridge uses the authorized manual **bridge**
anchor. Washington Monument uses the user-authorized **obelisk** correction (2026-11-15):
the supplied **altar** anchor produced 0 nonempty masks out of 50. The original 18/36 split
stays fixed, and obelisk uses H3's declared global-scalar fallback. The other 53 compatible
completed preparations are verified and reused without downloading their inputs again.
Numeric checks cannot establish semantic correctness; review every selected-view overlay.
Zero depth/validity maps are preserved as views without evaluation support, as in the reference evaluator.
Existing compatible masks, including completed scenes from the earlier notebook, are reused.


In [ ]:
import hashlib, json, os, subprocess, sys, time, zipfile
from pathlib import Path
_previous_source = globals().get('SOURCE')
from google.colab import drive
drive.mount('/content/drive')

SHARED = Path('/content/drive/MyDrive/Point3_Hypothesis54')
SOURCE_SHA256 = '83796943ffafa29f96a5fee1f4e58887a8b66b3da04515ebbd1375769ac4b446'
BUNDLE = SHARED / 'point3_shared54_source_83796943ffaf.zip'
CALIBRATION_SIZE = 18  # 18 calibration + 36 confirmation; 54 sweeps are optional.
RUN_LABEL = 'v1'       # Keep unchanged to resume; use a new label for a new study.
SESSION_HOURS = 8      # Session limit, NOT a predicted completion time.
DEVICES = '0'          # Use '0,1' only when two matching GPUs actually exist.
OUTPUT = SHARED / 'mask_preparation' / f'masks_{SOURCE_SHA256[:12]}_{RUN_LABEL}'
OUTPUT.mkdir(parents=True, exist_ok=True)
SOURCE = Path('/content') / ('point3_shared54_source_' + SOURCE_SHA256[:12])
if not BUNDLE.exists():
    raise FileNotFoundError(f'Keep the source ZIP in {SHARED} alongside this notebook: {BUNDLE.name}')
if hashlib.sha256(BUNDLE.read_bytes()).hexdigest() != SOURCE_SHA256:
    raise ValueError('Source bundle checksum mismatch')
with zipfile.ZipFile(BUNDLE) as archive:
    if archive.testzip() is not None: raise ValueError('Damaged source bundle')
    for entry in archive.infolist():
        dest = (SOURCE / entry.filename).resolve()
        if not dest.is_relative_to(SOURCE.resolve()): raise ValueError('Unsafe bundle path')
        payload = archive.read(entry)
        if dest.exists() and dest.read_bytes() != payload:
            raise ValueError(f'Local source changed: {dest}; use a separate version directory')
        dest.parent.mkdir(parents=True, exist_ok=True)
        if not dest.exists(): dest.write_bytes(payload)
STUDY = SOURCE / 'experiments/point3/shared54'
if _previous_source is not None and Path(_previous_source).resolve() != SOURCE.resolve():
    for _name, _module in list(sys.modules.items()):
        _file = getattr(_module, '__file__', None)
        if _file and Path(_file).resolve().is_relative_to(Path(_previous_source).resolve()):
            sys.modules.pop(_name, None)
sys.path[:0] = [str(STUDY), str(STUDY.parent), str(SOURCE)]
WORK = Path('/content/adaptiveattend_point3')
EVAL_PY = WORK / 'envs/vggt/bin/python'
SAM_PY = WORK / 'envs/sam3/bin/python'
SCRATCH = Path('/content/point3_mask_preparation_work')
DEADLINE = time.monotonic() + SESSION_HOURS*3600
print('Verified source:', SOURCE_SHA256[:12])
print('Persistent results:', OUTPUT)
print('54 landmarks; up to 50 eligible distinct photographs per landmark; actual counts recorded.')

COMMON_MASKS = SHARED/'common_masks'
MASK_SEARCH_ROOTS = [
    SHARED/'mask_preparation', SHARED/'results',
    Path('/content/drive/MyDrive/Point3_Shared54/results'),
    Path('/content/drive/MyDrive/AdaptiveAttend_Point3'),
    WORK/'results', WORK/'masks',
]
print('Shared masks for all hypotheses:', COMMON_MASKS)


In [ ]:
from protocol import make_plan, save
from mask_cache import discover, ready_path, require_ready
PLAN = make_plan()
MASKS_READY = False
if ready_path(COMMON_MASKS, PLAN).exists():
    READY = require_ready(COMMON_MASKS, PLAN, deep=True)
    MASKS_READY = True
    print('READY:', READY['landmarks'], 'landmarks /', READY['images'], 'masks')
    print('Original preparation and previews:', READY['preparation_output'])
else:
    found = discover(MASK_SEARCH_ROOTS)
    save(OUTPUT/'mask_discovery_preflight.json', found)
    print('Documented archives:', len(found['sources']))
    print('Unverified locations:', len(found['unverified']))
    if found['unverified']: print(json.dumps(found['unverified'][:10], indent=2))
    print('Coverage is established by matching actual image contents during preparation.')


## Install or reuse the SAM environment

Only SAM is installed. The token is read privately from Secrets when work starts.
A complete, verified cache skips setup and generation entirely.


In [ ]:
if MASKS_READY:
    print('All masks are already verified; no model installation needed.')
else:

    from gpu_setup import gpu_stack
    capabilities = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'], text=True, timeout=15).strip().splitlines()
    GPU_STACK = gpu_stack(capabilities)
    if GPU_STACK['blackwell']:
        EVAL_PY = WORK/'envs/vggt_cu128/bin/python'
        SAM_PY = WORK/'envs/sam3_cu128/bin/python'
    print('GPU library profile:', GPU_STACK)
    print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True, timeout=15))
    from protocol import save
    save(OUTPUT/'gpu_setup.json', GPU_STACK)

    from hypothesis_study import bounded_run, BudgetExceeded, write_json

    def checked(command, label):
        print(label, flush=True)
        return bounded_run(command, OUTPUT / 'setup.log', SOURCE, os.environ.copy(), DEADLINE)

    def pinned_repository(url, directory, revision):
        directory.parent.mkdir(parents=True, exist_ok=True)
        if not directory.exists():
            checked(['git', 'clone', '--quiet', url, directory], f'Cloning {directory.name}')
        # Read-only inspection; never overwrite a checkout with local edits.
        status = subprocess.check_output(['git', '-C', str(directory), 'status', '--porcelain'], text=True, timeout=10)
        if status.strip():
            raise RuntimeError(f'Local edits found in {directory}; resolve them before changing its revision')
        checked(['git', '-C', directory, 'fetch', '--quiet', 'origin', revision], f'Pinning {directory.name}')
        checked(['git', '-C', directory, 'checkout', '--quiet', revision], f'Checking {directory.name} revision')

    eval_probe = ("import torch,numpy,safetensors,sys,importlib;"
                  f"assert torch.__version__.split('+')[0]=={GPU_STACK['eval_torch']!r},torch.__version__;"
                  "assert numpy.__version__=='2.2.6',numpy.__version__;"
                  f"assert torch.version.cuda=={GPU_STACK['eval_cuda']!r},torch.version.cuda;"
                  "assert torch.cuda.is_available(),'Select a GPU runtime';torch.ones(2,device='cuda').sum().item();"
                  "sys.path[:0]=['intervention','.'];importlib.import_module('models.vggt.run_pass')")
    sam_probe = ("import torch,decord,pycocotools;from sam3.model_builder import build_sam3_image_model;"
                 "from sam3.model.sam3_image_processor import Sam3Processor;"
                 "assert torch.__version__.startswith('2.7.0'),torch.__version__;"
                 f"assert torch.version.cuda=={GPU_STACK['sam_cuda']!r},torch.version.cuda;"
                 "assert torch.cuda.is_available(),'SAM requires a GPU';torch.ones(2,device='cuda').sum().item()")

    def ready(python, probe, repo, revision):
        if not python.exists() or not (repo / '.git').exists():
            return False
        head = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True, timeout=10).strip()
        dirty = subprocess.check_output(['git', '-C', str(repo), 'status', '--porcelain'], text=True, timeout=10).strip()
        if head != revision or dirty:
            return False
        try:
            checked([python, '-c', probe], f'Checking existing {python.parent.parent.name} environment')
            return True
        except BudgetExceeded:
            raise
        except RuntimeError:
            return False

    vggt_revision = '97bbde571faddde3ace3cfa7724a20448026c4c8'
    sam_revision = '11dec2936de97f2857c1f76b66d982d5a001155d'
    vggt_dir, sam_dir = WORK / 'third_party/vggt', WORK / 'third_party/sam3'
    try:
        eval_ready = True
        sam_ready = ready(SAM_PY, sam_probe, sam_dir, sam_revision)
        if not (eval_ready and sam_ready):
            checked([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], 'Preparing isolated environments')
            uv = [sys.executable, '-m', 'uv']
            if not eval_ready:
                if not EVAL_PY.exists():
                    checked(uv + ['venv', '--python', '3.12', EVAL_PY.parent.parent], 'Creating VGGT environment')
                pip = uv + ['pip', 'install', '--python', EVAL_PY]
                checked(pip + ['torch=='+GPU_STACK['eval_torch'], 'torchvision=='+GPU_STACK['eval_vision'], '--index-url', 'https://download.pytorch.org/whl/cu'+GPU_STACK['eval_cuda'].replace('.','')], 'Installing VGGT GPU libraries')
                checked(pip + ['numpy==2.2.6', 'safetensors', '-r', SOURCE / 'requirements.txt'], 'Installing evaluation dependencies')
                pinned_repository('https://github.com/facebookresearch/vggt.git', vggt_dir, vggt_revision)
                checked(pip + ['--no-deps', '-e', vggt_dir], 'Installing VGGT')
            if not sam_ready:
                if not SAM_PY.exists():
                    checked(uv + ['venv', '--python', '3.12', SAM_PY.parent.parent], 'Creating SAM 3 environment')
                pip = uv + ['pip', 'install', '--python', SAM_PY]
                checked(pip + ['setuptools<81', 'wheel'], 'Installing SAM build dependencies')
                checked(pip + ['torch==2.7.0', 'torchvision==0.22.0', '--index-url', 'https://download.pytorch.org/whl/cu'+GPU_STACK['sam_cuda'].replace('.','')], 'Installing SAM GPU libraries')
                pinned_repository('https://github.com/facebookresearch/sam3.git', sam_dir, sam_revision)
                checked(pip + ['-e', sam_dir, 'einops', 'psutil', 'numpy<2', 'pillow', 'decord==0.6.0', 'pycocotools'], 'Installing SAM 3 dependencies')
        pass # No evaluation environment in mask preparation.
        checked([SAM_PY, '-c', sam_probe], 'Validating SAM imports and GPU')
        # Preserve actual package versions and include them in checkpoint identity.
        for name, python in [('sam', SAM_PY)]:
            versions_path = OUTPUT / (name+'_packages.json')
            script = ("import importlib.metadata,json,pathlib; "
                      f"pathlib.Path({str(versions_path)!r}).write_text(json.dumps(sorted((d.metadata['Name'],d.version) for d in importlib.metadata.distributions())))")
            bounded_run([python, '-c', script], OUTPUT / 'setup.log', SOURCE, os.environ.copy(), DEADLINE)
        print(f'Environments ready. {(DEADLINE-time.monotonic())/60:.1f} minutes remain in this session.')
    except Exception as exc:
        write_json(OUTPUT / 'status.json', {'status': 'setup_incomplete', 'reason': type(exc).__name__, 'num_views': 50})
        raise


## Import, generate missing masks, and resume

Downloads each landmark archive as needed, selects up to 50 eligible distinct images,
imports matching masks, generates missing ones, and saves validation and previews.
Completed landmarks are skipped on resume; partial generation continues from saved masks.
Any failure remains explicit and blocks the ready marker until all 54 landmarks succeed.


In [ ]:
if not MASKS_READY:
    from google.colab import userdata
    from hypothesis_study import bounded_run
    remaining = (DEADLINE-time.monotonic())/3600
    if remaining <= 0: raise RuntimeError('Session budget expired; rerun settings to resume.')
    environment = os.environ.copy()
    try:
        try:
            environment['HF_TOKEN'] = userdata.get('HF_TOKEN')
        except Exception:
            raise RuntimeError('Add HF_TOKEN in Colab Secrets, enable notebook access, and obtain SAM 3 approval.') from None
        environment['HF_HUB_DISABLE_IMPLICIT_TOKEN'] = '0'
        command = [SAM_PY, '-u', STUDY/'prepare_masks.py', '--output', OUTPUT, '--work', SCRATCH,
                   '--sam-python', SAM_PY, '--mask-cache', COMMON_MASKS, '--hours', str(remaining)]
        for root in MASK_SEARCH_ROOTS: command.extend(['--mask-search-root', root])
        bounded_run(command, OUTPUT/'notebook.log', SOURCE, environment, DEADLINE, echo=True)
    finally:
        environment.pop('HF_TOKEN', None)
READY = require_ready(COMMON_MASKS, PLAN, deep=True)
MASKS_READY = True
print('READY:', READY['landmarks'], 'landmarks /', READY['images'], 'masks')
print('H1, H2 and H3 can now reuse this cache:', COMMON_MASKS)


## Review coverage and masks

Each scene has a mask archive, input identities, numeric validation, reuse counts and preview.
Manual-anchor scenes (Forth Bridge and Washington Monument) show all selected views;
other scenes show up to six previews. Empty SAM outputs are retained
as valid no-detection records, but at least two views per scene must contain a detected anchor.
Review other scene previews in the printed preparation folder as well.


In [ ]:
from IPython.display import display, Image, Markdown
review_root = Path(READY['preparation_output']) if MASKS_READY else OUTPUT
summary = review_root/'summary.json'
info = {}
if summary.exists():
    info = json.loads(summary.read_text())
    print({k:v for k,v in info.items() if k not in ('entries','cohort')})
reuse = [json.loads(p.read_text()) for p in review_root.glob('scenes/*/mask_reuse.json')]
print('Saved per-scene mask counts (latest attempt):',
      {k:sum(r.get(k,0) for r in reuse) for k in ['reused','imported','generated']})
entries = READY['entries'] if MASKS_READY else info.get('entries', [])
counts = [{'scene':e['identity']['scene'], 'num_views':len(e['identity']['images'])} for e in entries]
display(counts)
print('Landmarks below the 50-view cap:', [r for r in counts if r['num_views'] < 50])
for scene, override in PLAN['anchor_overrides'].items():
    scene_id = hashlib.sha256(scene.encode()).hexdigest()[:16]
    preview = review_root/'scenes'/scene_id/'mask_preview.jpg'
    if preview.exists():
        display(Markdown(f"### {scene} — manual {override['anchor']} anchor; inspect all selected-view overlays"))
        display(Image(filename=str(preview)))
        print(json.loads((preview.parent/'mask_validation.json').read_text()))
print('All previews, scene logs and provenance:', review_root)
print('Shared reusable mask cache:', COMMON_MASKS)
